# Kinthread scoring on Kaggle
Inputs: private dataset with `real_questions_full.tgz`, and the r8 / r8p notebook outputs. Scores every candidate on the full real dev (fair + verified keys). The locked test runs only for `FINAL`, once.

In [ ]:
import os, glob, hashlib, subprocess, json
REPO = '/kaggle/temp/Kinthread-llm-model'
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    !git clone --depth 1 -b main https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
%cd {REPO}
tgz = glob.glob('/kaggle/input/**/real_questions_full.tgz', recursive=True)
assert tgz, 'attach the private dataset with real_questions_full.tgz'
!rm -rf data/real_questions && tar -xzf {tgz[0]}
print(len(glob.glob('data/real_questions/*.jsonl')), 'question files')
if not os.path.exists('runs/v10-r7/best.pt'):
    !cat runs/v10-r7/best.pt.part-* > runs/v10-r7/best.pt
assert hashlib.sha256(open('runs/v10-r7/best.pt', 'rb').read()).hexdigest() == open('runs/v10-r7/best.pt.sha256').read().strip()
CKPTS = {'r7': 'runs/v10-r7/best.pt'}
for p in glob.glob('/kaggle/input/**/best.pt', recursive=True):
    CKPTS[p.split('/runs/')[-1].replace('/best.pt', '').replace('v10-', '') + '@' + p.split('/')[3]] = p
FINAL = ''   # set to one CKPTS name to score the locked test once
print(CKPTS)

In [ ]:
W = '/kaggle/working'
SWEEP = ''   # one CKPTS name: also score it with the reading boosters
CONFIGS = {'base': ''}
if SWEEP:
    CONFIGS.update({'cover': '--cover', 'para2': '--para 2', 'k16': '--k 16', 'split': '--split_search', 'all': '--cover --para 2 --k 16 --split_search'})
for name, ck in CKPTS.items():
    for cfg, flags in (CONFIGS.items() if name == SWEEP else [('base', '')]):
        out = f'{W}/dev_{name.replace("@", "_")}_{cfg}.json'
        !python -m eval.run_real --ckpt {ck} --split dev --adaptive 1.0 --hybrid {flags} --out {out} | grep -E '"dev'
        print('^', name, cfg)

In [ ]:
FINAL_FLAGS = ''   # the dev-chosen booster flags for the final model
if FINAL:
    !python -m eval.run_real --ckpt {CKPTS[FINAL]} --split test --adaptive 1.0 --hybrid {FINAL_FLAGS} --out {W}/LOCKED_TEST_{FINAL.replace("@", "_")}.json | grep -E '"test'